In [ ]:
# Requirements
!pip install nibabel av

In [ ]:
import k3d
import nibabel as nib
import numpy as np
from k3d.helpers import download

In [ ]:
# The Visible Human cryosections are photographs, so the colour is the measurement rather
# than a scalar put through a colormap. NIfTI stores that as RGB24 (datatype 128), which nibabel
# surfaces as a structured dtype - one uint8 field per channel - so it needs a view before it is
# an ordinary array.
filename = download('https://raw.githubusercontent.com/neurolabusc/niivue-images/main/visiblehuman.nii.gz')
img = nib.load(filename)

packed = np.asanyarray(img.dataobj)                              # (x, y, z, 1), ('R', 'G', 'B')
rgb = packed.view(np.uint8).reshape(packed.shape[:3] + (3,))     # (x, y, z, 3)

# k3d indexes a volume as [z, y, x]; this file is RAS, so its first axis is x
rgb = np.ascontiguousarray(np.transpose(rgb, (2, 1, 0, 3)))

# 1 mm isotropic here, but read it off the header rather than assuming
dx, dy, dz = img.header.get_zooms()[:3]
nz, ny, nx = rgb.shape[:3]
bounds = [
    -nx * dx / 2, nx * dx / 2,
    -ny * dy / 2, ny * dy / 2,
    -nz * dz / 2, nz * dz / 2,
]

print(rgb.shape, rgb.dtype, '%.0f MB' % (rgb.nbytes / 1e6))

In [ ]:
# A slice shows the bytes it was given: no colormap, no window, nothing to tune.
# color_map and color_range are refused with a warning here, because there is nothing to map.
human = k3d.volume_slice(rgb,
                         slice_z=rgb.shape[0] // 2,
                         slice_y=rgb.shape[1] // 2,
                         slice_x=rgb.shape[2] // 2,
                         bounds=bounds)

plot = k3d.plot(camera_mode='volume_sides', grid_visible=False, background_color=0)
plot += human
plot.display()

In [ ]:
# Move the planes; the three views follow.
human.slice_z = int(rgb.shape[0] * 0.62)

In [ ]:
# The raymarch needs an alpha, and an RGB volume has no scalar to take it from, so it comes
# from Rec. 709 luminance shaped by opacity_function. Half of this volume is the air around the
# body (luminance under 0.05), which is what the flat start of the ramp removes - without it the
# box is opaque and all you see is its front face.
#
# Where the ramp rises matters more here than it would for a scalar field. A volume is sampled
# trilinearly, so every surface has a rim where the texture fades in. A scalar field hides that:
# whatever value the ray stops at, the colormap turns it into a full-intensity colour. Here the
# value IS the colour, so a ray that stops halfway up the rim paints a half-bright one. Starting
# the rise at 0.30 instead of just above the air puts the surface where the skin is already
# itself - measured on this volume, the skin goes from (70, 60, 46) to (83, 70, 54).
human_3d = k3d.volume(rgb,
                      opacity_function=[0.0, 0.0, 0.30, 0.0, 0.42, 1.0, 1.0, 1.0],
                      alpha_coef=120.0,
                      samples=512.0,
                      bounds=bounds)

plot = k3d.plot(grid_visible=False, background_color=0)
plot += human_3d
plot.camera_auto_fit = False
plot.camera = [500, -520, 260, 0, 0, 0, 0, 0, 1]
plot.display()

In [ ]:
# Do not expect a ramp to undress this. Skin and brain sit at the same luminance here, both
# around 0.4 to 0.55, so no shape of opacity_function can drop one and keep the other: what
# separates them is hue, and luminance is exactly the part of the colour that threw hue away.
# Cutting is the way in.
plot.clipping_planes = [[0, 1, 0, 0]]

In [ ]:
# Maximum intensity projection over the same volume. A maximum has to be a maximum of
# something, and luminance is the only scalar a colour has, so that is what the ray maximises -
# then the pixel keeps the colour of the voxel that reached it. The teeth and the vault of the
# skull come through the skin in the colours they actually have.
human_mip = k3d.mip(rgb, samples=512.0, bounds=bounds)

plot = k3d.plot(grid_visible=False, background_color=0)
plot += human_mip
plot.camera_auto_fit = False
plot.camera = [500, -520, 260, 0, 0, 0, 0, 0, 1]
plot.display()

In [ ]:
# A film is an RGB volume nobody calls one: (frames, height, width, 3) is exactly the shape
# k3d indexes as [z, y, x], with time where depth usually goes. Big Buck Bunny, CC-BY, Blender
# Foundation - ten seconds at 640x360.
import av

filename = download('https://test-videos.co.uk/vids/bigbuckbunny/mp4/h264/360/Big_Buck_Bunny_360_10s_1MB.mp4')

with av.open(filename) as container:
    frames = [frame.to_ndarray(format='rgb24') for frame in container.decode(video=0)]

# every third frame at half resolution: all 300 at full size would be 207 MB on the wire.
# rows are reversed because an image counts y downwards and k3d counts it up
film = np.ascontiguousarray(np.stack(frames)[::3, ::-2, ::2])
nz, ny, nx = film.shape[:3]
film_bounds = [0, nx, 0, ny, 0, nz]

print(film.shape, film.dtype, '%.0f MB' % (film.nbytes / 1e6))

In [ ]:
# Look down the time axis and the z slice is simply a frame.
movie = k3d.volume_slice(film, slice_z=nz // 2, slice_y=-1, slice_x=-1, bounds=film_bounds)

plot = k3d.plot(grid_visible=False, background_color=0)
plot += movie
plot.camera_auto_fit = False
plot.camera = [nx / 2, ny / 2, 700, nx / 2, ny / 2, nz / 2, 0, 1, 0]
plot.camera_fov = 20.0
plot.display()

In [ ]:
# Turn the block and the other two planes are slit-scans: one row, or one column, of pixels
# drawn against time. Every streak is a pixel that held its colour while the shot moved.
movie = k3d.volume_slice(film, slice_z=nz // 2, slice_y=ny // 2, slice_x=nx // 2,
                         bounds=film_bounds)

plot = k3d.plot(camera_mode='volume_sides', grid_visible=False, background_color=0)
plot += movie
plot.display()

In [ ]:
# Maximum along the time axis: for every pixel of the frame, the brightest it ever was.
# The radial smear is not a rendering artefact - the same maximum taken in numpy looks the same -
# it is the shot's own push-in, with each point of the scene dragged outwards as it approached.
plot = k3d.plot(grid_visible=False, background_color=0)
plot += k3d.mip(film, samples=512.0, bounds=film_bounds)
plot.camera_auto_fit = False
plot.camera = [nx / 2, ny / 2, 3000, nx / 2, ny / 2, nz / 2, 0, 1, 0]
plot.camera_fov = 5.0
plot.display()

In [ ]:
# A segmentation mask still works on top of RGB, and still multiplicatively - the tint scales
# the photographic colour instead of replacing it, so the tissue stays readable underneath.
# This one is synthetic, the brightest sixth of the tissue, standing in for a real label map.
luminance = rgb[..., 0] * 0.2126 + rgb[..., 1] * 0.7152 + rgb[..., 2] * 0.0722
mask = (luminance > 0.62 * 255).astype(np.uint8)

human = k3d.volume_slice(rgb,
                         slice_z=rgb.shape[0] // 2,
                         slice_y=rgb.shape[1] // 2,
                         slice_x=rgb.shape[2] // 2,
                         mask=mask,
                         active_masks=[1],
                         mask_opacity=0.6,
                         bounds=bounds)

plot = k3d.plot(camera_mode='volume_sides', grid_visible=False, background_color=0)
plot += human
plot.display()